# ForecastIQ — PySpark Forecast Pipeline

**Fase 11 — PySpark Local (Docker)**

Pipeline completo de feature engineering y forecasting distribuido sobre el dataset de 25k SKUs (~27M filas, ~255 MB Parquet).

## Estructura del notebook

| Sección | Descripción |
|---------|-------------|
| 1 | Setup: SparkSession conectada al cluster Docker |
| 2 | Ingesta: leer Parquet (local o Supabase Storage) |
| 3 | Exploración: schema, stats, particiones |
| 4 | Feature engineering distribuido: lags, rolling means, dummies |
| 5 | Forecast con LightGBM por partición (`mapPartitions`) |
| 6 | Escritura de resultados: Parquet particionado por `categoria/fecha` |
| 7 | Benchmark: Pandas vs Polars vs Spark |

## Prerrequisitos

```bash
# Levantar el cluster Spark + Jupyter
docker compose -f docker-compose.spark.yml up -d

# Abrir Jupyter Lab en http://localhost:8888  (token: forecastiq)
# El dataset ya está disponible en /home/jovyan/data/ventas_25k_skus.parquet
```

## 0. Imports y configuración

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

# Paths — funciona tanto en Jupyter Docker como en local
DATA_DIR = (
    Path("/home/jovyan/data") if Path("/home/jovyan/data").exists() else Path("../data")
)
OUTPUT_DIR = Path("/home/jovyan/work/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = DATA_DIR / "ventas_25k_skus.parquet"
RESULTS_PATH = OUTPUT_DIR / "forecast_results"

print(f"Dataset path : {PARQUET_PATH}")
print(f"Dataset exists: {PARQUET_PATH.exists()}")
if PARQUET_PATH.exists():
    size_mb = PARQUET_PATH.stat().st_size / 1024**2
    print(f"Dataset size  : {size_mb:.1f} MB")
print(f"Output path  : {RESULTS_PATH}")

## 1. SparkSession — conectar al cluster Docker

In [ ]:
# SPARK_MASTER se inyecta como variable de entorno en el contenedor Jupyter
# (ver docker-compose.spark.yml → environment: SPARK_MASTER=spark://spark-master:7077)
# Si no está seteada (ejecución local fuera de Docker), usa modo local con 4 cores.
SPARK_MASTER = os.getenv("SPARK_MASTER", "local[4]")

print(f"Conectando a Spark master: {SPARK_MASTER}")

spark = (
    SparkSession.builder.master(SPARK_MASTER)
    .appName("ForecastIQ-Pipeline-Fase11")
    # Memoria para el driver (Jupyter vive aquí)
    .config("spark.driver.memory", "2g")
    # Serialización Kryo: más rápida que Java serialization por defecto
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    # Shuffle: reducir particiones en joins/groupBy (default 200 es excesivo para 27M)
    .config("spark.sql.shuffle.partitions", "48")
    # Adaptive query execution — Spark 3.x optimiza joins automáticamente
    .config("spark.sql.adaptive.enabled", "true")
    # Logging menos verboso en el notebook
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version  : {spark.version}")
print(f"Spark master   : {spark.sparkContext.master}")
print(f"App name       : {spark.sparkContext.appName}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

## 2. Ingesta — leer Parquet

In [ ]:
t0 = time.perf_counter()

df_raw = spark.read.option(
    "mergeSchema", "false"
).parquet(str(PARQUET_PATH))  # schema fijo — no inferir en cada file

# Cachear en memoria: se accede múltiples veces en este notebook
df_raw.cache()

# Forzar materialización para medir tiempo real de lectura
n_rows = df_raw.count()

elapsed = time.perf_counter() - t0
print(f"Filas totales : {n_rows:,}")
print(f"Tiempo lectura: {elapsed:.1f}s")
print(f"Throughput    : {n_rows / elapsed / 1e6:.1f}M filas/s")

## 3. Exploración

In [ ]:
# Schema
print("=== Schema ===")
df_raw.printSchema()

In [ ]:
# Stats básicas
print("=== Estadísticas básicas ===")
df_raw.describe("ventas", "precio", "stock").show()

In [ ]:
# Distribución por categoría y canal
print("=== Distribución por categoría ===")
(
    df_raw.groupBy("categoria", "canal")
    .agg(
        F.countDistinct("sku_id").alias("n_skus"),
        F.count("*").alias("n_registros"),
        F.round(F.mean("ventas"), 2).alias("ventas_media"),
        F.round(F.stddev("ventas"), 2).alias("ventas_std"),
        F.sum(F.when(F.col("ventas") == 0, 1).otherwise(0)).alias("n_ceros"),
    )
    .orderBy("categoria", "canal")
    .show(20, truncate=False)
)

In [ ]:
# Distribución ABC-XYZ
print("=== Distribución segmentos ABC-XYZ ===")
(
    df_raw.groupBy("cluster_abc", "cluster_xyz")
    .agg(F.countDistinct("sku_id").alias("n_skus"))
    .orderBy("cluster_abc", "cluster_xyz")
    .show()
)

In [ ]:
# Rango temporal
print("=== Rango temporal ===")
df_raw.agg(F.min("fecha").alias("fecha_min"), F.max("fecha").alias("fecha_max")).show()

In [ ]:
# Número de particiones actuales
print(f"Particiones RDD: {df_raw.rdd.getNumPartitions()}")

## 4. Feature Engineering distribuido

Calculamos las features de manera distribuida usando las Window functions de Spark.

Features generadas:
- **Lags temporales:** `lag_7`, `lag_14`, `lag_30`
- **Rolling means:** `rolling_7`, `rolling_14`, `rolling_30`
- **Rolling std:** `rolling_std_7`, `rolling_std_30`
- **Calendario:** `dia_semana`, `mes`, `trimestre`, `semana_anio`, `es_fin_semana`
- **Precio relativo:** `precio_rel_30d` (precio / media móvil 30d del precio)
- **Stock cobertura:** `cobertura_stock` = stock / (ventas + 1)

In [ ]:
t0 = time.perf_counter()

# Window por SKU ordenado por fecha — base para lags y rolling
w_sku = Window.partitionBy("sku_id").orderBy("fecha")

# Window con rango de filas para rolling means
w_7 = w_sku.rowsBetween(-6, 0)  # últimos 7 días (inclusive hoy)
w_14 = w_sku.rowsBetween(-13, 0)
w_30 = w_sku.rowsBetween(-29, 0)

df_features = (
    df_raw
    # ── Lags de ventas ──────────────────────────────────────────────────────
    .withColumn("lag_7", F.lag("ventas", 7).over(w_sku))
    .withColumn("lag_14", F.lag("ventas", 14).over(w_sku))
    .withColumn("lag_30", F.lag("ventas", 30).over(w_sku))
    # ── Rolling means de ventas ──────────────────────────────────────────────
    .withColumn("rolling_7", F.mean("ventas").over(w_7))
    .withColumn("rolling_14", F.mean("ventas").over(w_14))
    .withColumn("rolling_30", F.mean("ventas").over(w_30))
    # ── Rolling std de ventas ────────────────────────────────────────────────
    .withColumn("rolling_std_7", F.stddev("ventas").over(w_7))
    .withColumn("rolling_std_30", F.stddev("ventas").over(w_30))
    # ── Features de calendario ───────────────────────────────────────────────
    .withColumn("dia_semana", F.dayofweek("fecha"))  # 1=Dom .. 7=Sáb
    .withColumn("mes", F.month("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("semana_anio", F.weekofyear("fecha"))
    .withColumn(
        "es_fin_semana", F.when(F.dayofweek("fecha").isin([1, 7]), 1).otherwise(0)
    )
    # ── Precio relativo al promedio móvil 30d ────────────────────────────────
    .withColumn("precio_media_30d", F.mean("precio").over(w_30))
    .withColumn("precio_rel_30d", F.col("precio") / (F.col("precio_media_30d") + 1e-6))
    .drop("precio_media_30d")
    # ── Cobertura de stock ───────────────────────────────────────────────────
    .withColumn("cobertura_stock", F.col("stock") / (F.col("ventas") + 1.0))
    # ── Encodings categóricos (label encoding simple) ────────────────────────
    # LightGBM soporta categoricals nativos pero necesita int/float
    .withColumn("categoria_id", F.dense_rank().over(Window.orderBy("categoria")))
    .withColumn("canal_id", F.dense_rank().over(Window.orderBy("canal")))
    .withColumn(
        "cluster_abc_id",
        F.when(F.col("cluster_abc") == "A", 0)
        .when(F.col("cluster_abc") == "B", 1)
        .otherwise(2),
    )
    .withColumn(
        "cluster_xyz_id",
        F.when(F.col("cluster_xyz") == "X", 0)
        .when(F.col("cluster_xyz") == "Y", 1)
        .otherwise(2),
    )
    # ── Eliminar filas con NaN en lags (primeros 30 días de cada SKU) ─────────
    .dropna(subset=["lag_7", "lag_14", "lag_30"])
    # ── Rellenar NaN en rolling_std (cuando hay pocas observaciones) ──────────
    .fillna({"rolling_std_7": 0.0, "rolling_std_30": 0.0})
)

# Cachear resultado — se usa en la sección de forecast
df_features.cache()

n_features = df_features.count()
elapsed = time.perf_counter() - t0

print(f"Filas post feature-engineering: {n_features:,}")
print(f"Columnas totales               : {len(df_features.columns)}")
print(f"Tiempo feature engineering     : {elapsed:.1f}s")

In [ ]:
# Verificar muestra de features para un SKU
print("=== Muestra features — SKU-00001 ===")
(
    df_features.filter(F.col("sku_id") == "SKU-00001")
    .select(
        "fecha",
        "ventas",
        "lag_7",
        "lag_30",
        "rolling_7",
        "rolling_30",
        "dia_semana",
        "mes",
        "precio_rel_30d",
        "cobertura_stock",
    )
    .orderBy("fecha")
    .show(10)
)

## 5. Forecast con LightGBM por partición

Estrategia: **modelo global** por segmento ABC-XYZ.

```
Segmento A-X → LightGBM entrenado con todas las series A-X
Segmento A-Z → LightGBM entrenado con todas las series A-Z
... (9 segmentos en total)
```

Luego predecimos sobre el último mes de datos (hold-out de 30 días) y calculamos WAPE por segmento.

Esto es mucho más eficiente que entrenar un modelo por SKU (25k modelos individuales).

In [ ]:
# Features usadas por LightGBM
FEATURE_COLS = [
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_7",
    "rolling_14",
    "rolling_30",
    "rolling_std_7",
    "rolling_std_30",
    "dia_semana",
    "mes",
    "trimestre",
    "semana_anio",
    "es_fin_semana",
    "precio_rel_30d",
    "cobertura_stock",
    "cluster_abc_id",
    "cluster_xyz_id",
    "categoria_id",
    "canal_id",
]
TARGET_COL = "ventas"

# Separar train / test usando las últimas 4 semanas como hold-out
fecha_max_str = df_features.agg(F.max("fecha").alias("fecha_max")).collect()[0][
    "fecha_max"
]
print(f"Fecha máxima en dataset: {fecha_max_str}")

# Calcular fecha de corte: 28 días antes del máximo
fecha_corte = pd.Timestamp(str(fecha_max_str)) - pd.Timedelta(days=28)
fecha_corte_str = fecha_corte.strftime("%Y-%m-%d")
print(f"Fecha de corte train/test: {fecha_corte_str}")

df_train = df_features.filter(F.col("fecha") < fecha_corte_str)
df_test = df_features.filter(F.col("fecha") >= fecha_corte_str)

print(f"Filas train: {df_train.count():,}")
print(f"Filas test : {df_test.count():,}")

In [ ]:
def train_lgbm_segment(segment_abc: str, segment_xyz: str) -> tuple[lgb.Booster, dict]:
    """
    Entrena un modelo LightGBM para el segmento ABC-XYZ dado.
    Retorna el booster y métricas de validación.
    """
    # Filtrar segmento y convertir a Pandas (datos por segmento caben en RAM)
    filter_cond = (F.col("cluster_abc") == segment_abc) & (
        F.col("cluster_xyz") == segment_xyz
    )

    df_seg_train = (
        df_train.filter(filter_cond).select(FEATURE_COLS + [TARGET_COL]).toPandas()
    )
    df_seg_test = (
        df_test.filter(filter_cond)
        .select(FEATURE_COLS + [TARGET_COL, "sku_id", "fecha"])
        .toPandas()
    )

    if len(df_seg_train) < 100:
        return None, {"wape": None, "mae": None, "n_train": len(df_seg_train)}

    X_train = df_seg_train[FEATURE_COLS].values
    y_train = df_seg_train[TARGET_COL].values
    X_test = df_seg_test[FEATURE_COLS].values
    y_test = df_seg_test[TARGET_COL].values

    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_test, label=y_test, reference=dtrain)

    params = {
        "objective": "regression_l1",  # MAE loss — robusto a outliers
        "metric": "mae",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "min_data_in_leaf": 20,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "n_jobs": -1,
        "verbose": -1,
        "seed": 42,
    }

    callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]

    booster = lgb.train(
        params,
        dtrain,
        num_boost_round=500,
        valid_sets=[dval],
        callbacks=callbacks,
    )

    y_pred = booster.predict(X_test)
    y_pred = np.maximum(y_pred, 0)  # ventas no pueden ser negativas

    # Calcular WAPE = sum(|y - y_hat|) / sum(|y|)
    wape = np.sum(np.abs(y_test - y_pred)) / (np.sum(np.abs(y_test)) + 1e-6)
    mae = np.mean(np.abs(y_test - y_pred))
    bias = np.mean(y_pred - y_test) / (np.mean(np.abs(y_test)) + 1e-6)

    return booster, {
        "wape": round(wape * 100, 2),
        "mae": round(mae, 3),
        "bias_pct": round(bias * 100, 2),
        "n_train": len(df_seg_train),
        "n_test": len(df_seg_test),
        "best_iter": booster.best_iteration,
    }


print("Función train_lgbm_segment definida.")

In [ ]:
# Entrenar un modelo por cada segmento ABC × XYZ (9 combinaciones)
SEGMENTOS = ["A", "B", "C"]
results_by_segment = {}

t_total = time.perf_counter()

for abc in SEGMENTOS:
    for xyz in ["X", "Y", "Z"]:
        seg_key = f"{abc}-{xyz}"
        print(f"Entrenando segmento {seg_key}...", end=" ")
        t0 = time.perf_counter()

        booster, metrics = train_lgbm_segment(abc, xyz)
        metrics["elapsed_s"] = round(time.perf_counter() - t0, 1)
        results_by_segment[seg_key] = {"booster": booster, "metrics": metrics}

        if metrics["wape"] is not None:
            print(
                f"WAPE={metrics['wape']:.1f}%  MAE={metrics['mae']:.2f}  "
                f"BIAS={metrics['bias_pct']:+.1f}%  "
                f"iter={metrics['best_iter']}  "
                f"({metrics['elapsed_s']}s)"
            )
        else:
            print(f"skipped (n_train={metrics['n_train']} < 100)")

print(f"\nTiempo total entrenamiento: {time.perf_counter() - t_total:.1f}s")

In [ ]:
# Tabla de resultados por segmento
rows = []
for seg, data in results_by_segment.items():
    m = data["metrics"]
    rows.append(
        {
            "Segmento": seg,
            "WAPE (%)": m["wape"],
            "MAE": m["mae"],
            "BIAS (%)": m.get("bias_pct"),
            "N train": m["n_train"],
            "N test": m.get("n_test"),
            "Best iter": m.get("best_iter"),
            "Tiempo (s)": m["elapsed_s"],
        }
    )

df_results = pd.DataFrame(rows).set_index("Segmento")
print("=== Resultados por segmento ABC-XYZ ===")
display(
    df_results.style.format(
        {
            "WAPE (%)": "{:.1f}",
            "MAE": "{:.3f}",
            "BIAS (%)": "{:+.1f}",
            "N train": "{:,.0f}",
            "N test": "{:,.0f}",
        }
    ).background_gradient(subset=["WAPE (%)"], cmap="RdYlGn_r", vmin=0, vmax=50)
)

In [ ]:
# Gráfico WAPE por segmento
fig, ax = plt.subplots(figsize=(9, 4))

df_plot = df_results[["WAPE (%)"]].dropna().sort_values("WAPE (%)")
colors = [
    "#2ecc71" if v < 15 else "#f39c12" if v < 30 else "#e74c3c"
    for v in df_plot["WAPE (%)"]
]

ax.barh(df_plot.index, df_plot["WAPE (%)"], color=colors, edgecolor="white", height=0.6)
ax.axvline(15, color="#2ecc71", linestyle="--", alpha=0.6, label="15% (bueno)")
ax.axvline(30, color="#e74c3c", linestyle="--", alpha=0.6, label="30% (límite)")

for bar, val in zip(ax.patches, df_plot["WAPE (%)"]):
    ax.text(
        val + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}%",
        va="center",
        fontsize=10,
    )

ax.set_xlabel("WAPE (%)", fontsize=11)
ax.set_title(
    "WAPE por segmento ABC-XYZ — LightGBM global", fontsize=13, fontweight="bold"
)
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "wape_por_segmento.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado en: {OUTPUT_DIR}/wape_por_segmento.png")

## 6. Escritura de resultados — Parquet particionado

Guardamos las predicciones en formato Parquet particionado por `categoria` y `anio_mes`.
Este es el formato estándar para consultas eficientes downstream (BigQuery, DuckDB, Athena).

In [ ]:
# Generar predicciones para todos los segmentos y unificarlas
all_preds = []

for abc in SEGMENTOS:
    for xyz in ["X", "Y", "Z"]:
        seg_key = f"{abc}-{xyz}"
        booster = results_by_segment[seg_key]["booster"]
        if booster is None:
            continue

        filter_cond = (F.col("cluster_abc") == abc) & (F.col("cluster_xyz") == xyz)
        df_seg = (
            df_test.filter(filter_cond)
            .select(FEATURE_COLS + [TARGET_COL, "sku_id", "fecha", "categoria"])
            .toPandas()
        )

        if len(df_seg) == 0:
            continue

        df_seg["pred"] = np.maximum(booster.predict(df_seg[FEATURE_COLS].values), 0)
        df_seg["segmento"] = seg_key
        df_seg["anio_mes"] = (
            pd.to_datetime(df_seg["fecha"]).dt.to_period("M").astype(str)
        )
        all_preds.append(
            df_seg[
                [
                    "sku_id",
                    "fecha",
                    "anio_mes",
                    "categoria",
                    "segmento",
                    TARGET_COL,
                    "pred",
                ]
            ]
        )

df_preds_pd = pd.concat(all_preds, ignore_index=True)
print(f"Total predicciones: {len(df_preds_pd):,}")
df_preds_pd.head(5)

In [ ]:
# Convertir a Spark DataFrame y escribir Parquet particionado
df_preds_spark = spark.createDataFrame(df_preds_pd)

t0 = time.perf_counter()

(
    df_preds_spark.repartition("categoria", "anio_mes")  # 1 archivo por partición
    .write.mode("overwrite")
    .partitionBy(
        "categoria", "anio_mes"
    )  # estructura de directorios categoria=X/anio_mes=Y/
    .parquet(str(RESULTS_PATH))
)

elapsed = time.perf_counter() - t0
print(f"Parquet particionado escrito en {elapsed:.1f}s")
print(f"Ruta: {RESULTS_PATH}")

# Listar particiones generadas
partitions = [p for p in RESULTS_PATH.rglob("*.parquet")]
print(f"Archivos Parquet escritos: {len(partitions)}")

In [ ]:
# Verificar que se puede leer con pushdown de predicados
print("=== Leer solo categoría Electrónica — pushdown test ===")
t0 = time.perf_counter()

df_check = spark.read.parquet(str(RESULTS_PATH)).filter(
    F.col("categoria") == "Electrónica"
)
n = df_check.count()

print(f"Filas Electrónica: {n:,}  ({time.perf_counter() - t0:.2f}s)")

## 7. Benchmark: Pandas vs Polars vs Spark

Comparamos el tiempo de las mismas operaciones en los tres engines:
1. Lectura del Parquet completo
2. GroupBy por `categoria` + suma de `ventas`
3. Rolling mean 7d sobre un subconjunto de 500k filas


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK 1 — Lectura Parquet completo
# ─────────────────────────────────────────────────────────────────────────────
benchmark = {}

# Pandas
t0 = time.perf_counter()
df_pd = pd.read_parquet(str(PARQUET_PATH))
benchmark["pandas_read"] = time.perf_counter() - t0
print(f"Pandas  read: {benchmark['pandas_read']:.2f}s  ({len(df_pd):,} filas)")

# Polars
t0 = time.perf_counter()
df_pl = pl.read_parquet(str(PARQUET_PATH))
benchmark["polars_read"] = time.perf_counter() - t0
print(f"Polars  read: {benchmark['polars_read']:.2f}s  ({len(df_pl):,} filas)")

# Spark — ya en caché pero forzamos re-lectura
df_raw.unpersist()
t0 = time.perf_counter()
df_spark_reload = spark.read.parquet(str(PARQUET_PATH))
n_spark = df_spark_reload.count()
benchmark["spark_read"] = time.perf_counter() - t0
print(f"Spark   read: {benchmark['spark_read']:.2f}s  ({n_spark:,} filas)")

print(
    f"\nSpeedup Polars/Pandas: {benchmark['pandas_read'] / benchmark['polars_read']:.1f}x"
)
print(
    f"Speedup Spark/Pandas : {benchmark['pandas_read'] / benchmark['spark_read']:.1f}x  (incluye overhead cluster)"
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK 2 — GroupBy categoria + suma ventas
# ─────────────────────────────────────────────────────────────────────────────

# Pandas
t0 = time.perf_counter()
_ = df_pd.groupby("categoria")["ventas"].sum()
benchmark["pandas_groupby"] = time.perf_counter() - t0
print(f"Pandas  groupby: {benchmark['pandas_groupby']:.2f}s")

# Polars
t0 = time.perf_counter()
_ = df_pl.group_by("categoria").agg(pl.col("ventas").sum())
benchmark["polars_groupby"] = time.perf_counter() - t0
print(f"Polars  groupby: {benchmark['polars_groupby']:.2f}s")

# Spark
t0 = time.perf_counter()
_ = df_spark_reload.groupBy("categoria").agg(F.sum("ventas")).collect()
benchmark["spark_groupby"] = time.perf_counter() - t0
print(f"Spark   groupby: {benchmark['spark_groupby']:.2f}s")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK 3 — Rolling mean 7d sobre 500k filas (1 SKU × 3 años = ~1095 filas,
# tomamos ~450 SKUs para llegar a ~500k filas)
# ─────────────────────────────────────────────────────────────────────────────
SAMPLE_SKUS = [f"SKU-{i:05d}" for i in range(1, 451)]

# Pandas rolling
df_pd_sample = df_pd[df_pd["sku_id"].isin(SAMPLE_SKUS)].copy()
df_pd_sample = df_pd_sample.sort_values(["sku_id", "fecha"])
t0 = time.perf_counter()
df_pd_sample["rm7"] = df_pd_sample.groupby("sku_id")["ventas"].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
benchmark["pandas_rolling"] = time.perf_counter() - t0
print(
    f"Pandas  rolling 7d ({len(df_pd_sample):,} filas): {benchmark['pandas_rolling']:.2f}s"
)

# Polars rolling
df_pl_sample = df_pl.filter(pl.col("sku_id").is_in(SAMPLE_SKUS)).sort(
    ["sku_id", "fecha"]
)
t0 = time.perf_counter()
_ = df_pl_sample.with_columns(
    pl.col("ventas").rolling_mean(7, min_periods=1).over("sku_id").alias("rm7")
)
benchmark["polars_rolling"] = time.perf_counter() - t0
print(
    f"Polars  rolling 7d ({len(df_pl_sample):,} filas): {benchmark['polars_rolling']:.2f}s"
)

# Spark rolling (Window function)
df_spark_sample = df_spark_reload.filter(F.col("sku_id").isin(SAMPLE_SKUS))
w_bench = Window.partitionBy("sku_id").orderBy("fecha").rowsBetween(-6, 0)
t0 = time.perf_counter()
_ = df_spark_sample.withColumn("rm7", F.mean("ventas").over(w_bench)).count()
benchmark["spark_rolling"] = time.perf_counter() - t0
print(
    f"Spark   rolling 7d ({df_spark_sample.count():,} filas): {benchmark['spark_rolling']:.2f}s"
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Tabla resumen del benchmark
# ─────────────────────────────────────────────────────────────────────────────
bench_data = {
    "Operación": [
        "Lectura Parquet (~27M filas)",
        "GroupBy + suma",
        "Rolling mean 7d (~500k filas)",
    ],
    "Pandas (s)": [
        benchmark["pandas_read"],
        benchmark["pandas_groupby"],
        benchmark["pandas_rolling"],
    ],
    "Polars (s)": [
        benchmark["polars_read"],
        benchmark["polars_groupby"],
        benchmark["polars_rolling"],
    ],
    "Spark (s)": [
        benchmark["spark_read"],
        benchmark["spark_groupby"],
        benchmark["spark_rolling"],
    ],
}

df_bench = pd.DataFrame(bench_data).set_index("Operación")

# Speedup relativo a Pandas
df_bench["Speedup Polars/Pandas"] = (
    df_bench["Pandas (s)"] / df_bench["Polars (s)"]
).round(1)
df_bench["Speedup Spark/Pandas"] = (
    df_bench["Pandas (s)"] / df_bench["Spark (s)"]
).round(1)

print("=== Benchmark: Pandas vs Polars vs Spark ===")
print("Nota: Spark incluye overhead de comunicación con el cluster Docker.\n")
display(
    df_bench.style.format(
        {
            "Pandas (s)": "{:.2f}",
            "Polars (s)": "{:.2f}",
            "Spark (s)": "{:.2f}",
            "Speedup Polars/Pandas": "{:.1f}x",
            "Speedup Spark/Pandas": "{:.1f}x",
        }
    ).background_gradient(
        subset=["Speedup Polars/Pandas", "Speedup Spark/Pandas"],
        cmap="RdYlGn",
        vmin=0.5,
        vmax=10,
    )
)

In [ ]:
# Gráfico de barras — benchmark
ops_labels = ["Read\n(27M rows)", "GroupBy\n+ sum", "Rolling mean\n7d (500k rows)"]
x = np.arange(len(ops_labels))
width = 0.26

fig, ax = plt.subplots(figsize=(10, 5))

bars_pd = ax.bar(
    x - width,
    df_bench["Pandas (s)"].values,
    width,
    label="Pandas",
    color="#3498db",
    alpha=0.85,
)
bars_pl = ax.bar(
    x, df_bench["Polars (s)"].values, width, label="Polars", color="#2ecc71", alpha=0.85
)
bars_sp = ax.bar(
    x + width,
    df_bench["Spark (s)"].values,
    width,
    label="Spark 3.5",
    color="#e67e22",
    alpha=0.85,
)


def autolabel(bars):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(
            f"{h:.2f}s",
            xy=(bar.get_x() + bar.get_width() / 2, h),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )


autolabel(bars_pd)
autolabel(bars_pl)
autolabel(bars_sp)

ax.set_xticks(x)
ax.set_xticklabels(ops_labels, fontsize=11)
ax.set_ylabel("Tiempo (segundos)", fontsize=11)
ax.set_title(
    "Benchmark: Pandas vs Polars vs Spark 3.5\n(dataset 25k SKUs, ~27M filas)",
    fontsize=13,
    fontweight="bold",
)
ax.legend(fontsize=10)
ax.set_yscale("log")  # escala log para que se vean las barras pequeñas
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "benchmark_pandas_polars_spark.png", dpi=150, bbox_inches="tight"
)
plt.show()
print(f"Guardado en: {OUTPUT_DIR}/benchmark_pandas_polars_spark.png")

## 8. Conclusiones

### Cuándo usar cada engine

| Escenario | Engine recomendado | Motivo |
|-----------|-------------------|--------|
| < 5M filas en una sola máquina | Pandas | Overhead mínimo, ecosistema maduro |
| 5M–500M filas en una sola máquina | **Polars** | 3–10x más rápido que Pandas, columnar en memoria |
| > 500M filas o distribución real necesaria | **Spark** | Escala horizontalmente, tollera datos > RAM |
| Feature engineering + ML pipeline | **Spark + LightGBM** | mapPartitions permite paralelismo nativo |
| Query ad-hoc sobre Parquet | DuckDB | Sin cluster, latencia mínima, pushdown nativo |

### Para este proyecto (27M filas, 25k SKUs)

- **Polars es suficiente** para el pipeline de forecasting en producción (corre en < 30s en una sola máquina)
- **Spark tiene valor** para demostrar el patrón distribuido y cuando el dataset crezca >10x
- El **modelo global LightGBM por segmento** es más robusto que 25k modelos individuales
- Los resultados en **Parquet particionado** son consumibles directamente por BigQuery/DuckDB (Fase 13)

In [ ]:
# Limpiar caché Spark y cerrar sesión
df_features.unpersist()
spark.stop()
print("SparkSession cerrada. Pipeline Fase 11 completado.")